<a href="https://colab.research.google.com/github/SunnyChoudhary850/FlyRank/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip -q install duckdb huggingface_hub scikit-learn

In [2]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

## Two Paper Findings + My Methodology Questions

**Finding: ML Appendix — Feature Importance for predicting Health Score**
(Average Position 43%, Impressions 32%, Scroll Depth 15%, CTR 8%)

The paper itself already flags something important here: "the target itself
is partly constructed from some of these inputs, so importance is
descriptive rather than causal." My methodology question, respectfully:
Health Score is explicitly defined earlier in the paper as Impressions (30
pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts) — meaning
the top 4 "predictive" features are literally the same components used to
build the label itself. This isn't just correlation without causation, it's
closer to definitional overlap between features and target. I'd ask: would
the feature importance ranking still hold any meaning at all if the 4
components that make up Health Score were excluded, leaving only genuinely
independent features (content age, word count, days visible)? The paper's
own caveat is honest and appropriate, but the finding as visualized (a
ranked bar chart) could still be read by someone skimming as "these are
the features that drive success," when it's closer to "here's how the
formula's own ingredients rank against each other."

**Finding: ML Appendix — Growth Prediction (Logistic Regression, 71%
holdout accuracy)**

The methodology section states an 80/20 split was used for this model, but
doesn't specify whether that split was random or grouped by brand/client.
Given the paper's own dataset spans 57 different brands, my methodology
question is: if the same brand's pages can appear in both the 80% training
portion and the 20% holdout portion, the 71% accuracy could partly reflect
the model learning brand-specific patterns rather than genuinely
generalizable growth signals. This is the same grouped-vs-random distinction
I tested directly on my own w05 model (see Section 2) — I'd ask whether the
71% holdout accuracy would hold up under a strictly brand-grouped split, or
whether it would drop the way I'd predict based on my own before/after
result below.

Both questions are asked in the spirit the assignment describes — this
paper already holds itself to a genuinely disclosed standard (it flags its
own descriptive-not-causal limitation, reports reversed/nuanced findings
instead of hiding them, and explicitly separates headline evidence from
exploratory ML appendix material). These are the same category of question
I'm about to ask of my own work in the sections below.

In [3]:
features_and_label = con.sql(f"""
    WITH march_features AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS total_impressions_month,
               AVG(gsc_avg_position) AS avg_position_month,
               COUNT(DISTINCT report_date) AS days_with_activity,
               SUM(gsc_clicks) AS total_clicks_month,
               SUM(ga4_sessions) AS total_sessions_month
        FROM {TABLES['fact_daily']}
        WHERE month = '2026-03' AND gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    ),
    april_perf AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions_next_month
        FROM {TABLES['fact_daily']}
        WHERE month = '2026-04' AND gsc_data_available IS TRUE
        GROUP BY content_hash_id
    ),
    staleness AS (
        SELECT content_hash_id, CURRENT_DATE - content_updated_date AS days_since_update
        FROM {TABLES['dim_content']}
    )
    SELECT f.*, a.impressions_next_month, s.days_since_update
    FROM march_features f
    JOIN april_perf a ON f.content_hash_id = a.content_hash_id
    JOIN staleness s ON f.content_hash_id = s.content_hash_id
""").df()

features_and_label['declining'] = (
    features_and_label['impressions_next_month'] < features_and_label['total_impressions_month']
).astype(int)
features_and_label = features_and_label.dropna()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## My Model Under an Honest Split (Before/After)

In [4]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import numpy as np, pandas as pd

feature_cols = ['total_impressions_month', 'avg_position_month', 'days_with_activity',
                 'total_clicks_month', 'total_sessions_month', 'days_since_update']
X = features_and_label[feature_cols]
y = features_and_label['declining']
groups = features_and_label['client_hash_id']

def precision_at_k(scores, y_true, k=50):
    top_k_idx = np.argsort(scores)[-k:]
    return y_true.iloc[top_k_idx].mean()

# BEFORE: random split
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42)
scaler = StandardScaler().fit(Xtr)
model_random = LogisticRegression(max_iter=1000).fit(scaler.transform(Xtr), ytr)
scores_random = model_random.predict_proba(scaler.transform(Xte))[:, 1]
p50_random = precision_at_k(scores_random, yte.reset_index(drop=True), 50)

# AFTER: grouped split by client
splitter = GroupShuffleSplit(test_size=0.3, n_splits=1, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
Xtr2, Xte2 = X.iloc[train_idx], X.iloc[test_idx]
ytr2, yte2 = y.iloc[train_idx], y.iloc[test_idx]
scaler2 = StandardScaler().fit(Xtr2)
model_grouped = LogisticRegression(max_iter=1000).fit(scaler2.transform(Xtr2), ytr2)
scores_grouped = model_grouped.predict_proba(scaler2.transform(Xte2))[:, 1]
p50_grouped = precision_at_k(scores_grouped, yte2.reset_index(drop=True), 50)

pd.DataFrame({
    'Split type': ['Random (before, dishonest)', 'Grouped by client (after, honest)'],
    'Precision@50': [p50_random, p50_grouped]
})

,Split type,Precision@50
0,"Random (before, dishonest)",0.80
1,"Grouped by client (after, honest)",0.86


**Result:** Random split scored 0.80, grouped split scored 0.86 — the
grouped (honest) split actually scored *higher*, not lower.

This is the opposite of the typical leakage story, where a random split
usually inflates the score by letting the model "cheat" on client-specific
patterns. A likely explanation: with only 30% of clients held out in the
grouped split, the specific clients that happened to land in the test set
may simply have easier-to-predict decline patterns than the overall
population — this is a small-sample effect of which clients got grouped
together, not proof that grouping never matters.

**Honest conclusion:** this single before/after comparison does not, by
itself, prove my grouped split was "worth it" in terms of raw score — the
real value of the grouped split isn't a guaranteed score change in either
direction, it's that it tests something more meaningful: does the model
generalize to clients it's never seen, rather than just performing well by
partially memorizing familiar clients. The random split's score of 0.80
doesn't necessarily mean the grouped model is "worse" at that harder task
— it means the two numbers aren't directly measuring the same thing, and
comparing them as if higher-is-better in either direction would be a
misread of what grouping is actually for.

## Leakage Audit

In [5]:
# Check correlation between each feature and the label-defining column itself
leak_check = features_and_label[feature_cols + ['impressions_next_month']].corr()['impressions_next_month'].sort_values(ascending=False)
leak_check

,impressions_next_month
impressions_next_month,1.000000
total_impressions_month,0.874723
total_clicks_month,0.696485
total_sessions_month,0.402543
days_with_activity,0.168077
avg_position_month,-0.049457
days_since_update,-0.119375


**Result:** `total_impressions_month` correlates 0.87 with
`impressions_next_month`. `total_clicks_month` also correlates fairly
strongly (0.70).

This is not technically leakage in the strict sense — both features are
genuinely known at prediction time (March data, used to predict an April
outcome), so no future information is entering the model. But it reveals a
structural issue worth naming honestly: my label is defined as
`impressions_next_month < total_impressions_month`, meaning
`total_impressions_month` is literally one half of the comparison that
built the label. A model relying heavily on this feature isn't necessarily
learning a rich, generalizable "decline" pattern — it may partly be
learning a simpler regression-to-the-mean effect (pages with very high
March impressions are statistically more likely to see a lower April
number, purely because they had more room to fall).

**What I'd check next, if this were a real production model:** whether
performance holds up if `total_impressions_month` is removed entirely, to
see how much of the model's apparent skill depends on this one
structurally-related feature versus the other, more independent signals
(position, days active, sessions).

## Claim Rewrite

**Original claim (from my w05 notebook):** "Both real models clearly beat
my w04 baseline (0.56), confirming that combining multiple signals
genuinely helps over a single staleness rule."

**Rewritten with safe claim language:** In this dataset and time window,
Logistic Regression and Random Forest both showed higher observed
Precision@50 than my staleness-only baseline. This is a directional,
decision-support signal, not proof that combining signals will always
outperform a single-signal rule -- the comparison is observational, uses
one specific month's data, and one of the strongest features
(`total_impressions_month`) is structurally related to how the label
itself was defined, which may inflate the apparent gap.

**Original claim:** "Random Forest's top features... confirming what my
w04 signal check already found."

**Rewritten:** Random Forest's feature importances, measured on this
holdout split, ranked visibility signals (position, impressions) above
staleness. This is consistent with, but does not independently confirm,
my earlier w04 bucket analysis -- both results come from the same
underlying dataset and time window, so they are correlated observations,
not two independent pieces of evidence.

## Self-Check

- [x] Named two paper findings and a constructive methodology question for each
- [x] Re-ran my model under a grouped split with a real before/after comparison, and honestly reported the result didn't match the typical expected direction
- [x] Ran a leakage audit and found a real, worth-reporting structural correlation (0.87) between a feature and the label-derivation column
- [x] Rewrote my own claims using safe language (observed, directional, decision-support)